
# Bank Account Fraud Detection
## Stage 1 – Dataset Study

### Research Question

What does performing well mean when approximately
99% of bank account applications are legitimate?

### Notebook Objective

This notebook documents the initial study of the
Bank Account Fraud (BAF) Base dataset.

The analysis follows the project documentation
and focuses on:

1. Understanding the dataset and its features.
2. Identifying the target variable.
3. Examining missing-value sentinel conventions.
4. Understanding class imbalance.
5. Preparing a chronological data split.
6. Identifying potential data leakage risks.

### Experimental Protocol

The project uses the following chronological split:

- Training: months 0–4
- Validation: month 5
- Held-out test: months 6–7

The held-out test period will not be inspected
or used during Stage 1 model development.

### Documentation Approach

Each analysis step will include:

- Objective
- Implementation
- Observed results
- Interpretation
- Limitations or open questions

All findings will be documented after running
the corresponding analysis.

### Reproducibility

The notebook will record the dataset version,
Python environment, preprocessing decisions,
and relevant experimental settings.



## 1. Official Dataset Documentation

### 1.1 Objective

Before performing any data analysis, we review the
official Bank Account Fraud (BAF) dataset documentation.

The purpose of this step is to understand:

- How the dataset was created.
- The meaning of the target variable.
- The temporal structure of the dataset.
- The categorical features.
- Missing-value conventions.
- Potential limitations and data leakage risks.

### 1.2 Dataset Overview

**Dataset name:** Bank Account Fraud (BAF) — Base variant

**Dataset type:** Synthetic bank account application data.

**Prediction task:** Binary classification.

The goal is to predict whether a bank account
application is fraudulent or legitimate.

The BAF dataset suite was developed for research
on fraud detection and related machine learning
challenges.

### 1.3 Target Variable

The target variable is `fraud_bool`.

- `0`: Legitimate application.
- `1`: Fraudulent application.

The dataset is highly imbalanced, with fraudulent
applications representing a small minority.

Therefore, classification accuracy alone may
provide a misleading evaluation of performance.

### 1.4 Temporal Feature

The dataset includes a feature named `month`,
representing the application month.

The available month values range from 0 to 7.

For this project, the chronological split is:

- **Training:** Months 0–4.
- **Validation:** Month 5.
- **Held-out test:** Months 6–7.

The held-out test period must not be inspected
during Stage 1 model development.

This split follows the project protocol.

### 1.5 Categorical Features

The dataset contains five categorical features:

1. `payment_type`
2. `employment_status`
3. `housing_status`
4. `source`
5. `device_os`

These features require appropriate encoding
before being used by machine learning models.

### 1.6 Missing-Value Conventions

Some numerical features use negative values,
such as `-1`, to indicate missing information.

Therefore, checking only for standard NaN
values is not sufficient.

Sentinel values must be identified and
interpreted before preprocessing.

### 1.7 Dataset Limitations

The dataset is synthetic rather than a direct
collection of real banking applications.

As a result, model performance on this dataset
may not fully generalize to real-world fraud
detection systems.

Potential data leakage must also be considered,
especially when using temporally ordered data.

### 1.8 Questions for Further Investigation

1. What is the exact fraud rate in training months 0–4?
2. Does the fraud rate change in validation month 5?
3. Which numerical features contain sentinel values?
4. Are there features that could introduce data leakage?
5. How should categorical variables be encoded?
6. Which evaluation metrics are appropriate for
   highly imbalanced fraud detection?

### 1.9 References

Official BAF Dataset Datasheet:

https://raw.githubusercontent.com/feedzai/bank-account-fraud/main/documents/datasheet.pdf

Project Dataset Study:

https://github.com/fidaaarrabi/ml-project-bank-fraud-detection/blob/main/docs/dataset_study.md

### 1.10 Conclusion

The BAF Base dataset presents a binary
classification problem with severe class imbalance.

Understanding the dataset documentation,
missing-value conventions, and chronological
structure is essential before developing
fraud detection models.

The next step is to load the dataset and
verify its structure using Python.


In [2]:

# =====================================================
# 2. Environment Setup and Reproducibility
# =====================================================

import sys
import pandas as pd
import numpy as np
import sklearn

print("=== ENVIRONMENT INFORMATION ===")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")


=== ENVIRONMENT INFORMATION ===
Python version: 3.12.10
Pandas version: 3.1.1
NumPy version: 2.5.5
Scikit-learn version: 1.9.1



## 3. Dataset Loading

### Objective

Load the BAF Base dataset for Stage 1 analysis
while respecting the chronological data protocol.

### Chronological Split

- Training: months 0–4.
- Validation: month 5.
- Held-out test: months 6–7.

Only observations from months 0–5 will be
retained for Stage 1 analysis.

### Data Storage

The dataset is stored locally at
`data/raw/Base.csv`.

Raw data files are excluded from Git version
control using the `.gitignore` file.


In [3]:

# =====================================================
# 3.1 Dataset Loading
# =====================================================

from pathlib import Path

# Define the dataset path
data_path = Path("../data/raw/Base.csv")

# Support execution from the project root
if not data_path.is_file():
    data_path = Path("data/raw/Base.csv")

# Verify that the dataset exists
assert data_path.is_file(), "Dataset file not found."

# Load and retain only months 0–5
df = pd.concat(
    (
        chunk.loc[chunk["month"].between(0, 5)]
        for chunk in pd.read_csv(
            data_path,
            chunksize=100_000
        )
    ),
    ignore_index=True
)

# Verify that only Stage 1 months are retained
assert not df.empty, "No observations were loaded."
assert df["month"].between(0, 5).all()

# Display loading results
print("=== DATASET LOADING ===")
print("Dataset loaded successfully.")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Available months: {sorted(df['month'].unique().tolist())}")


=== DATASET LOADING ===
Dataset loaded successfully.
Number of rows: 794,989
Number of columns: 32
Available months: [0, 1, 2, 3, 4, 5]



### 3.2 Loading Results and Interpretation

The dataset was successfully loaded and restricted
to the Stage 1 period.

The resulting dataset contains 794,989 observations
and 32 columns.

Only months 0–5 are retained, confirming that
the held-out test months 6–7 are excluded
from the working dataset.

This restriction helps prevent information leakage
from the held-out test period during model development.



## 4. Dataset Shape and Types

### Objective

Examine the structure of the Stage 1 dataset,
identify the target variable, and distinguish
between numerical and categorical features.

This step provides the foundation for subsequent
data analysis and preprocessing decisions.


In [4]:

# =====================================================
# 4.1 Dataset Structure and Feature Types
# =====================================================

# Display dataset dimensions
print("=== DATASET SHAPE ===")
print(f"Number of observations: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Display column names and data types
print("\n=== COLUMN NAMES AND DATA TYPES ===")
print(df.dtypes)

# Define the target variable
target_column = "fraud_bool"

# Verify that the target exists
assert target_column in df.columns, "Target column not found."

# Separate predictors from the target
features = df.drop(columns=[target_column])

# Identify numerical features
numerical_features = features.select_dtypes(
    include="number"
).columns.tolist()

# Identify categorical features
categorical_features = features.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Display feature summary
print("\n=== FEATURE SUMMARY ===")
print(f"Target variable: {target_column}")
print(f"Total features: {features.shape[1]}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

print("\n=== NUMERICAL FEATURES ===")
print(numerical_features)

print("\n=== CATEGORICAL FEATURES ===")
print(categorical_features)


=== DATASET SHAPE ===
Number of observations: 794,989
Number of columns: 32

=== COLUMN NAMES AND DATA TYPES ===
fraud_bool                            int64
income                              float64
name_email_similarity               float64
prev_address_months_count             int64
current_address_months_count          int64
customer_age                          int64
days_since_request                  float64
intended_balcon_amount              float64
payment_type                            str
zip_count_4w                          int64
velocity_6h                         float64
velocity_24h                        float64
velocity_4w                         float64
bank_branch_count_8w                  int64
date_of_birth_distinct_emails_4w      int64
employment_status                       str
credit_risk_score                     int64
email_is_free                         int64
housing_status                          str
phone_home_valid                      int64
phone_m


### 4.2 Dataset Structure — Findings and Interpretation

The Stage 1 dataset contains 794,989 observations
and 32 columns, including the target variable
`fraud_bool`.

The 31 non-target columns consist of:

- 26 numerical features.
- 5 categorical features.

The identified categorical features are consistent
with the official dataset documentation.

### Implications for Preprocessing

- Categorical features require appropriate encoding
  before logistic regression can be applied.
- Numerical features may require scaling depending
  on their meaning and distribution.
- Binary indicators should be distinguished from
  continuous numerical features.
- The `month` column is used for chronological
  splitting and should be evaluated separately
  from the predictive features.

### Conclusion

The dataset structure was successfully verified.

No feature encoding, scaling, feature removal,
or model training has been performed at this stage.

The next step is to examine the target distribution
in the training period and validation month.
